# 06 - Exact Sparse SNF & Julia Backend

Notebook 05 introduced the Smith Normal Form as the computational backbone of homology. Here we go one level deeper: the `math_core` convenience wrapper is fast and usually correct, but it does not *certify* its output. In surgery theory, an uncertified torsion coefficient is not a mathematical fact — it is a numerical coincidence.

This notebook covers `pysurgery.core.exact_snf_julia`, which adds:
- **Modular rank certification** (independent prime witnesses for rank)
- **p-adic CRT reconstruction** (exact diagonal without full transformation matrices)
- **Julia leaf-peeling** (O(V+E) sparse reduction via Markowitz ordering)
- **Batch mode** (one Julia session, many matrices)

Every result carries `exact=True` and a `theorem_tag` when certification succeeds.

## Learning Goals
- Understand why uncertified SNF can silently mis-classify torsion
- Use `compute_exact_sparse_snf` with `backend='auto'|'julia'|'python'`
- Interpret `ExactSNFResult` and `ModularRankCertificate` Pydantic contracts
- Use `compute_padic_snf_diagonal` for prime-local torsion extraction
- Use `compute_batch_snf` for multi-matrix pipelines

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | CW/simplicial complex with cells | `SimplicialComplex`, `CWComplex` |
| **Algebraic** | Boundary matrices $\partial_k$ | `boundary_matrix(k)` |
| **Result** | Smith diagonal = torsion orders + rank | `compute_exact_sparse_snf` → `ExactSNFResult` |

## Formal Grounding

For $A \in M_{m\times n}(\mathbb{Z})$ there exist invertible $U, V$ over $\mathbb{Z}$ with $UAV = \text{diag}(d_1,\ldots,d_r,0,\ldots)$ and $d_i \mid d_{i+1}$. The $d_i$ are the **invariant factors** — they are unique and determine the homology completely. Exact computation certifies that $d_i$ are the *true* invariant factors, not floating-point approximations.

In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from pysurgery.core.exact_snf_julia import (
    compute_exact_sparse_snf,
    compute_modular_rank_certificate,
    compute_padic_snf_diagonal,
    compute_batch_snf,
    ExactSNFResult,
    ModularRankCertificate,
    BatchSNFResult,
)
from pysurgery.core.math_core import get_sparse_snf_diagonal
from pysurgery.core.complexes import SimplicialComplex
from pysurgery.bridge.julia_bridge import julia_engine

if julia_engine.available:
    print("Julia engine available — exact backend active.")
    julia_engine.warmup()
else:
    print("Julia not available — falling back to certified Python backend.")

print("=" * 70)
print("06 - Exact Sparse SNF & Julia Backend: Setup Complete")
print("=" * 70)

## Part 1: Why Exact Matters — A Failure Mode

The `get_sparse_snf_diagonal` function in `math_core` uses a heuristic pivot selection that can collapse distinct torsion factors when the matrix entries are large. The example below has invariant factors `[1, 4, 6]` (torsion $\mathbb{Z}/4\mathbb{Z} \oplus \mathbb{Z}/6\mathbb{Z}$). A heuristic reduction may return `[1, 2, 12]` (torsion $\mathbb{Z}/2\mathbb{Z} \oplus \mathbb{Z}/12\mathbb{Z}$) — both have the same product but *different* group structure.

The `exact_snf_julia` module resolves this by checking divisibility at each step.

In [ ]:
# A matrix whose torsion is deliberately ambiguous for heuristic pivoting
A_torsion = sp.csr_matrix(np.array([
    [4, 0, 0],
    [0, 6, 0],
    [0, 0, 1],
], dtype=np.int64))

# Heuristic (may or may not be correct depending on pivot order)
heuristic_diag = get_sparse_snf_diagonal(A_torsion)
print(f"Heuristic diagonal: {heuristic_diag}")

# Exact (always correct, carries a certificate)
exact_result: ExactSNFResult = compute_exact_sparse_snf(
    A_torsion, backend="python", p_cert=[2, 3]
)
print(f"Exact diagonal:     {exact_result.diagonal}")
print(f"exact=True?         {exact_result.exact}")
print(f"theorem_tag:        {exact_result.theorem_tag}")
print(f"backend_used:       {exact_result.backend_used}")

# Interpretation
print()
print("Torsion group from exact result:")
for d in exact_result.torsion_orders:
    print(f"  Z/{d}Z")

## Part 2: The `ExactSNFResult` Contract

`compute_exact_sparse_snf` returns an `ExactSNFResult` Pydantic model. Every field has a precise mathematical meaning:

| Field | Type | Meaning |
|---|---|---|
| `diagonal` | `List[int]` | Full Smith diagonal including leading 1s |
| `rank` | `int` | Number of non-zero diagonal entries |
| `torsion_orders` | `List[int]` | Non-unit entries $d_i > 1$; these are the torsion orders |
| `exact` | `bool` | `True` iff certified by modular witnesses |
| `backend_used` | `str` | `'julia'` or `'python'` |
| `theorem_tag` | `str` | Unique identifier for the algorithm theorem |
| `rank_certificate` | `ModularRankCertificate \| None` | Prime-witness rank proof |

The `exact` flag is the key quality signal. Downstream surgery computations should check it before trusting results.

In [ ]:
# Demonstration on the torus T²: expected H₁(T²) = Z² (rank 2, no torsion)
T2 = SimplicialComplex.torus()
boundary_2 = T2.boundary_matrix(2)

result: ExactSNFResult = compute_exact_sparse_snf(
    boundary_2,
    backend="auto",
    p_cert=[2, 3, 5, 7],
)

print("ExactSNFResult for ∂₂(T²):")
print(f"  diagonal:       {result.diagonal}")
print(f"  rank:           {result.rank}")
print(f"  torsion_orders: {result.torsion_orders}")
print(f"  exact:          {result.exact}")
print(f"  backend_used:   {result.backend_used}")
print(f"  theorem_tag:    {result.theorem_tag}")

# Also check ∂₁ for free rank
boundary_1 = T2.boundary_matrix(1)
result_1 = compute_exact_sparse_snf(boundary_1, backend="auto")
print()
print("ExactSNFResult for ∂₁(T²):")
print(f"  rank: {result_1.rank}, torsion: {result_1.torsion_orders}")

# Derive H₁(T²) = ker(∂₁) / im(∂₂)
n_1_chains = boundary_1.shape[1]
free_rank_H1 = (n_1_chains - result_1.rank) - result.rank
print()
print(f"H₁(T²) free rank = {free_rank_H1}  (expected 2)")
print(f"H₁(T²) torsion   = {result.torsion_orders}  (expected none)")

## Part 3: Modular Rank Certification

The key insight behind efficient exact SNF: the **rank** of a matrix over $\mathbb{Z}$ equals its rank over $\mathbb{F}_p = \mathbb{Z}/p\mathbb{Z}$ for all but finitely many primes $p$. If `rank_mod_2 = rank_mod_3 = rank_mod_5 = r`, then the integral rank is $r$ with certainty.

`compute_modular_rank_certificate` returns a `ModularRankCertificate` that formalizes this agreement across primes. If `all_agree=True`, the rank is *certified*.

In [ ]:
cert: ModularRankCertificate = compute_modular_rank_certificate(
    boundary_2,
    prime_list=[2, 3, 5, 7, 11],
    backend="auto",
)

print("ModularRankCertificate for ∂₂(T²):")
print(f"  certified_rank:  {cert.certified_rank}")
print(f"  prime_witnesses: {cert.prime_witnesses}")
print(f"  all_agree:       {cert.all_agree}")
print(f"  exact:           {cert.exact}")
print(f"  theorem_tag:     {cert.theorem_tag}")
print()

# Visualise prime witness agreement
primes = list(cert.prime_witnesses.keys())
ranks  = list(cert.prime_witnesses.values())

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(primes, ranks, color="steelblue", edgecolor="k", width=0.6)
ax.axhline(cert.certified_rank, color="crimson", lw=2, ls="--", label=f"certified rank = {cert.certified_rank}")
ax.set_xlabel("Prime p"); ax.set_ylabel("rank mod p")
ax.set_title("Modular Rank Agreement across Primes")
ax.set_xticks(primes)
ax.legend()
plt.tight_layout(); plt.show()

## Part 4: p-Adic SNF Diagonal

For very large boundary matrices, computing the full transformation matrices $U, V$ is expensive. The **p-adic diagonal** reads off the $p^k$-part of each invariant factor directly, without constructing $U$ and $V$.

**Algorithm:** Reduce $A \bmod p^k$ iteratively using Hensel lifting. The result is the diagonal of $A$ in $\mathbb{Z}/p^k\mathbb{Z}$, which captures all $p$-primary torsion of order $\leq p^{k-1}$.

This is most useful for extracting 2-torsion ($p=2$) and 3-torsion ($p=3$) separately, then combining via CRT.

In [ ]:
# Klein bottle: H₁(K) = Z ⊕ Z/2Z — the 2-torsion is the key feature
K = SimplicialComplex.klein_bottle()
bK = K.boundary_matrix(1)

# 2-adic diagonal (precision = 4 means mod 2⁴ = 16)
padic_2 = compute_padic_snf_diagonal(bK, prime=2, precision=4)
print(f"2-adic diagonal of ∂₁(K):  {padic_2}")
print("(2s confirm Z/2Z torsion)")

# 3-adic diagonal — should be trivial for Klein bottle
padic_3 = compute_padic_snf_diagonal(bK, prime=3, precision=4)
print(f"3-adic diagonal of ∂₁(K):  {padic_3}")
print("(all 1s confirm no 3-torsion)")

# Cross-check with full exact SNF
full = compute_exact_sparse_snf(bK, backend="python", p_cert=[2, 3, 5])
print()
print(f"Full exact diagonal: {full.diagonal}")
print(f"Torsion orders:      {full.torsion_orders}")

## Part 5: Batch Mode for Large Complexes

When processing a multi-dimensional complex, you need the SNF of every boundary map $\partial_0, \partial_1, \ldots, \partial_n$. Calling `compute_exact_sparse_snf` individually starts a new Julia session for each call. `compute_batch_snf` processes all matrices in one session, minimising startup overhead.

**When to use:** any time you compute homology of a complex with $\geq 3$ non-trivial boundary maps.

In [ ]:
# CP² has boundary maps ∂₁, ∂₂, ∂₃, ∂₄ (4-dimensional complex)
CP2 = SimplicialComplex.complex_projective_space(2)
matrices = [CP2.boundary_matrix(k) for k in range(1, 5)]

print("Matrix sizes for CP²:")
for k, M in enumerate(matrices, start=1):
    print(f"  ∂_{k}: shape {M.shape}, nnz={M.nnz}")

batch: BatchSNFResult = compute_batch_snf(
    matrices,
    backend="auto",
    parallel=True,
)

print()
print("Batch SNF results:")
for k, res in enumerate(batch.results, start=1):
    print(f"  ∂_{k}: rank={res.rank:3d}, torsion={res.torsion_orders}, exact={res.exact}")

# Derive full homology
print()
print("Homology of CP² (should be Z in degrees 0, 2, 4):")
n_cells = [CP2.num_cells(k) for k in range(5)]
print(f"  H₀: free rank = {n_cells[0] - batch.results[0].rank}")
for k in range(1, 4):
    r_next = batch.results[k].rank if k < len(batch.results) else 0
    r_curr = batch.results[k-1].rank
    free = n_cells[k] - r_curr - r_next
    tors = batch.results[k-1].torsion_orders
    print(f"  H_{k}: free rank = {free}, torsion = {tors}")

## Part 6: Backend Selection Logic

The `backend='auto'` heuristic applies the following decision tree:

```
backend='auto'
├── Julia available?  → use 'julia' (leaf-peeling + Markowitz ordering)
│     └── matrix nnz < 100?  → use dense Julia SNF
│           └── matrix nnz ≥ 100?  → use sparse Julia SNF
└── Julia unavailable?  → use 'python' (_snf_diagonal_python_dense)
```

**When to force `backend='python'`:** unit tests, CI environments without Julia, and small matrices where Julia startup cost dominates.

**When to force `backend='julia'`:** production runs on large complexes where exactness and speed both matter.

In [ ]:
from pysurgery.core.exact_snf_julia import compute_exact_sparse_snf
from pysurgery.core.exceptions import BackendUnavailableError

M_small = T2.boundary_matrix(1)

# Force Python backend — always works, no Julia required
r_py = compute_exact_sparse_snf(M_small, backend="python")
print(f"Python backend: exact={r_py.exact}, backend_used='{r_py.backend_used}'")

# Force Julia backend — raises if Julia not configured
try:
    r_jl = compute_exact_sparse_snf(M_small, backend="julia")
    print(f"Julia  backend: exact={r_jl.exact}, backend_used='{r_jl.backend_used}'")
except (BackendUnavailableError, Exception) as e:
    print(f"Julia backend not available: {type(e).__name__}")
    print("  To enable: from pysurgery.bridge.julia_bridge import julia_engine")
    print("  Then run:  julia_engine.warmup()")

# Auto (recommended)
r_auto = compute_exact_sparse_snf(M_small, backend="auto")
print(f"Auto   backend: exact={r_auto.exact}, backend_used='{r_auto.backend_used}'")

## Part 7: Connecting to the Homology Pipeline

The practical workflow is:
1. Build your complex
2. Extract boundary matrices with `boundary_matrix(k)`
3. Run `compute_batch_snf` with `backend='auto'`
4. Read off `rank` and `torsion_orders` from each `ExactSNFResult`
5. Check `exact=True` before reporting results

This replaces the `math_core` pipeline used in NB 05 with certified output.

In [ ]:
def certified_homology(complex_, max_dim=4):
    """Compute certified homology groups for all dimensions up to max_dim."""
    matrices = []
    for k in range(1, max_dim + 1):
        try:
            M = complex_.boundary_matrix(k)
            matrices.append((k, M))
        except Exception:
            break

    if not matrices:
        return {}

    batch = compute_batch_snf([m for _, m in matrices], backend="auto")

    groups = {}
    n_cells_k = {k: complex_.num_cells(k) for k in range(max_dim + 1)}

    for idx, (k, _) in enumerate(matrices):
        rank_k   = batch.results[idx].rank
        rank_km1 = batch.results[idx - 1].rank if idx > 0 else 0
        tors     = batch.results[idx - 1].torsion_orders if idx > 0 else []
        free     = n_cells_k.get(k, 0) - rank_km1 - rank_k
        groups[k - 1] = {"free_rank": max(free, 0), "torsion": tors,
                         "exact": batch.results[idx].exact}
    return groups

# Test on S³
S3 = SimplicialComplex.sphere(3)
H_S3 = certified_homology(S3, max_dim=3)
print("H*(S³):")
for k, v in sorted(H_S3.items()):
    print(f"  H_{k}: free={v['free_rank']}, torsion={v['torsion']}, exact={v['exact']}")
print()
# Expected: H₀=Z, H₃=Z, all others 0

## Failure Modes

| Failure | Cause | Fix |
|---|---|---|
| `BackendUnavailableError` | Julia not installed or not warmed up | Run `julia_engine.warmup()` or use `backend='python'` |
| `exact=False` in result | Primes disagreed on rank — possible near-singular matrix | Add more primes to `p_cert`, or check matrix for all-zero rows |
| `torsion_orders` wrong | Matrix not over $\mathbb{Z}$ (float entries) | Cast input to `dtype=np.int64` before passing |
| Slow performance | `backend='python'` on large matrix | Switch to `backend='julia'` or use `compute_batch_snf` |

In [ ]:
# Failure mode: float matrix passed to exact SNF
A_float = np.array([[2.0, 0.0], [0.0, 3.0]])
try:
    bad = compute_exact_sparse_snf(sp.csr_matrix(A_float), backend="python")
    print(f"Result (may be wrong): {bad.diagonal}, exact={bad.exact}")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

# Fix: cast to int
A_int = sp.csr_matrix(A_float.astype(np.int64))
good = compute_exact_sparse_snf(A_int, backend="python")
print(f"Corrected result: {good.diagonal}, exact={good.exact}")

## Summary Checklist

- [x] Understood why heuristic SNF can mis-classify torsion groups
- [x] Used `compute_exact_sparse_snf` with `p_cert` to get `ExactSNFResult`
- [x] Read `exact`, `rank`, `torsion_orders`, and `theorem_tag` from results
- [x] Used `compute_modular_rank_certificate` to get prime-witness rank proof
- [x] Used `compute_padic_snf_diagonal` for prime-local torsion
- [x] Used `compute_batch_snf` for multi-matrix homology pipelines
- [x] Understood the `backend='auto'|'julia'|'python'` selection logic

## Next Steps
- **NB 07**: Intersection forms use certified SNF internally — you now know why
- **NB 23**: Spectral sequences call `compute_batch_snf` on each page differential
- **NB 29**: Rational surgery reconstruction uses `compute_padic_snf_diagonal` across primes